# Project: BBLF AI Selector v2
# Section: Optimisation & Simulation Tool
# Sub Section: During Tourny (Post Rnd 4)

Goal: Select the optimal starting 12 players prior to the start of the tournament 

Things to add:
1. Additional constraints to capture fantasy nuances e.g. captain points, vice captain trick 


# 0. Prerequistes

In [14]:
# pip install --upgrade pip

In [15]:
# !pip install mip==1.16rc0
!pip install mip

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [16]:
# 0. Prerequistes

import pandas as pd
import numpy as np
import os
import random
from mip import Model, xsum, maximize, BINARY 
from scipy.stats import norm
import joblib
from concurrent.futures import ProcessPoolExecutor
import warnings
warnings.filterwarnings("ignore")

# Import intermediary functions
from Optimisation_Functions_dur_rnd import roll_rnd_price_fn, optimise_fn_efp, optimise_fn_sim_fp, optimise_setup_fn

# Set working directory
os.getcwd()
directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2'
add_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/post_round_4/'
over_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/overall'
py_data_score_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/python_data/scoring'
py_data_score_rnd_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/python_data/scoring/post-round-4/'

In [17]:
# Optimisation Strategy
opt_strat = 'sim' # Options: 'efp' (new player expected fantasy points appraoch), 'sim' (new player fantasy point distribution approach)
current_rnd = 5 # Current Round of the Tournament
exp_pts_model = 'all' # Options: 'bat_bowl' (batting + bowling model), 'all' (combined model)
efp_budget = 2010600
sim_budget = 1994700

# Constraints
# squad_players = 15 Not required as strategy set to 15 players (following round 1 optimisation strategy)
locked_teams = ['Hobart Hurricanes', 'Sydney Thunder', 'Brisbane Heat', 'Melbourne Stars'] # Teams that can't be selected for current round

# For Simulation Optimisation Strategy
conf_int = 0.2
sim_num = 5000

# 1. New EFP Approach

# a. Data Extraction 

In [18]:
# New Season Optimisation Strategy (Expected Fantasy Points)
if opt_strat == 'efp':
    
    # Data Extraction 
    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026_efp_rnd_4_pg21.csv'), low_memory=False)

    # Create Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team","On_Bench","Trade_In", "Trade_Out"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
    # team_fix_df = team_fix_df[team_fix_df.Round >= current_rnd].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df , team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected points csv file
    if exp_pts_model == 'bat_bowl':
        efp_df = pd.read_csv(os.path.join(py_data_score_rnd_directory,'bbl15_efp_bat_bowl_model_score_short_pr4.csv'), low_memory=False)
    elif exp_pts_model == 'all':
        efp_df = pd.read_csv(os.path.join(py_data_score_rnd_directory,'bbl15_efp_model_score_short_pr4.csv'), low_memory=False)

    # Join on expected points for each game
    player_df_raw = pd.merge(game_df, efp_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")
    # If pts_type is exp then add fielding points to expected points
    player_df_raw["exp_points"] = np.where((player_df_raw["Role"] == "WK") & (player_df_raw["pts_type"] == "exp"), player_df_raw["exp_pts"] + 11.68,
                                           np.where((player_df_raw["pts_type"] == "exp"), player_df_raw["exp_pts"] + 4.42, player_df_raw["exp_pts"]))
    
        
    # Lock teams: Set Available=0 for players from locked teams in current round only (in team players should be unaffected)
    # Create flag for locked teams in current round
    player_df_raw['Lock_f'] = np.where((player_df_raw['Team'].isin(locked_teams)) & (player_df_raw['Round'] == current_rnd), 1, 0)
    print(f"Locked teams for Round {current_rnd}: {locked_teams}")

    # Set Available=0 for players from locked teams in current round only (in team players should be unaffected)
    player_df_raw.loc[(player_df_raw['Team'].isin(locked_teams)) & (player_df_raw['Round'] == current_rnd) & (player_df_raw['In_Team'] != 1), 'Available'] = 0

    # Set trade in players to available
    player_df_raw.loc[(player_df_raw['Trade_In'] == 1) & (player_df_raw['Round'] == current_rnd), 'Available'] = 1

    # Set trade out players to unlocked
    player_df_raw.loc[(player_df_raw['Trade_Out'] == 1) & (player_df_raw['Round'] == current_rnd), 'Lock_f'] = 0

    # Drop unnecessary columns
    player_df_raw = player_df_raw[["Name", "Team", "Role", "Price", "Wk_f", "Bat_f", "Bowl_f", "Available", "In_Team","On_Bench","Trade_In", "Trade_Out", "Round", "opp", "venue", "Home_f","Lock_f", "exp_points"]]
    player_df_raw["weight"] = 1

    # Create game count per player
    player_df_raw["game_num"] = player_df_raw.groupby("Name").cumcount() + 1


    
else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

Change opt_strat to efp for player efp Optimisation Strategy


# b. Player price prediction for each round & data prep for optimisation

In [19]:
if opt_strat == 'efp':
    # Incorporate Player Actual Points into dataframes used for pricing model

    
    # Import Pricing Models
    # Load Model Objects
    price_model_obj_1 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_1_game'))
    price_model_obj_2 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_2_game'))
    price_model_obj_3 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_3_game'))

    # Create rolling round player price dataframes
    player_df_r1, player_df_r2, player_df_r3, player_df_r4, player_df_r5, player_df_r6, player_df_r7, player_df_r8, player_df_r9 = roll_rnd_price_fn(player_df_raw, price_df, current_rnd, price_model_obj_1, price_model_obj_2, price_model_obj_3)

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

Change opt_strat to efp for player efp Optimisation Strategy


# c. Optimisation Process (During Tournament Selection) - Starting 12 + 3 Bench Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 12 + 3 bench player
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total budget of team is less than $1,802,500 (As bench costs $79,000)

## Optimisation Setup

In [20]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    # a. EFP Optimisation Variables Setup
    # points_r1, price_r1, weight_r1, in_team_r1, available_r1, wk_weight_r1, bat_weight_r1, bowl_weight_r1, \
    #     play_cnt_r1, total_player_r1, wk_cnt_r1, total_wk_r1, bat_cnt_r1, total_bat_r1, bowl_cnt_r1, total_bowl_r1, \
    #     budget_r1, total_budget_r1, player_df_r1, cnt_r1, max_player_r1, \
    # points_r2, price_r2, weight_r2, in_team_r2, available_r2, wk_weight_r2, bat_weight_r2, bowl_weight_r2, \
    # play_cnt_r2, total_player_r2, wk_cnt_r2, total_wk_r2, bat_cnt_r2, total_bat_r2, bowl_cnt_r2, total_bowl_r2, \
    # budget_r2, total_budget_r2, team_play_cnt_r2, total_team_player_r2, player_df_r2, cnt_r2, max_player_r2, \
    # points_r3, price_r3, weight_r3, in_team_r3, available_r3, wk_weight_r3, bat_weight_r3, bowl_weight_r3, \
    # play_cnt_r3, total_player_r3, wk_cnt_r3, total_wk_r3, bat_cnt_r3, total_bat_r3, bowl_cnt_r3, total_bowl_r3, \
    # budget_r3, total_budget_r3, team_play_cnt_r3, total_team_player_r3, player_df_r3, cnt_r3, max_player_r3, \
    # points_r4, price_r4, weight_r4, in_team_r4, available_r4, wk_weight_r4, bat_weight_r4, bowl_weight_r4, \
    # play_cnt_r4, total_player_r4, wk_cnt_r4, total_wk_r4, bat_cnt_r4, total_bat_r4, bowl_cnt_r4, total_bowl_r4, \
    # budget_r4, total_budget_r4, team_play_cnt_r4, total_team_player_r4, player_df_r4, cnt_r4, max_player_r4, \
    points_r5, price_r5, weight_r5, in_team_r5, on_bench_r5, lock_f_r5, trade_in_r5, trade_out_r5, available_r5, wk_weight_r5, bat_weight_r5, bowl_weight_r5, \
    play_cnt_r5, total_player_r5, wk_cnt_r5, total_wk_r5, bat_cnt_r5, total_bat_r5, bowl_cnt_r5, total_bowl_r5, \
    budget_r5, total_budget_r5, team_play_cnt_r5, total_team_player_r5, player_df_r5, cnt_r5, max_player_r5, \
    points_r6, price_r6, weight_r6, in_team_r6, available_r6, wk_weight_r6, bat_weight_r6, bowl_weight_r6, \
    play_cnt_r6, total_player_r6, wk_cnt_r6, total_wk_r6, bat_cnt_r6, total_bat_r6, bowl_cnt_r6, total_bowl_r6, \
    budget_r6, total_budget_r6, team_play_cnt_r6, total_team_player_r6, player_df_r6, cnt_r6, max_player_r6, \
    points_r7, price_r7, weight_r7, in_team_r7, available_r7, wk_weight_r7, bat_weight_r7, bowl_weight_r7, \
    play_cnt_r7, total_player_r7, wk_cnt_r7, total_wk_r7, bat_cnt_r7, total_bat_r7, bowl_cnt_r7, total_bowl_r7, \
    budget_r7, total_budget_r7, team_play_cnt_r7, total_team_player_r7, player_df_r7, cnt_r7, max_player_r7, \
    points_r8, price_r8, weight_r8, in_team_r8, available_r8, wk_weight_r8, bat_weight_r8, bowl_weight_r8, \
    play_cnt_r8, total_player_r8, wk_cnt_r8, total_wk_r8, bat_cnt_r8, total_bat_r8, bowl_cnt_r8, total_bowl_r8, \
    budget_r8, total_budget_r8, team_play_cnt_r8, total_team_player_r8, player_df_r8, cnt_r8, max_player_r8, \
    points_r9, price_r9, weight_r9, in_team_r9, available_r9, wk_weight_r9, bat_weight_r9, bowl_weight_r9, \
    play_cnt_r9, total_player_r9, wk_cnt_r9, total_wk_r9, bat_cnt_r9, total_bat_r9, bowl_cnt_r9, total_bowl_r9, \
    budget_r9, total_budget_r9, team_play_cnt_r9, total_team_player_r9, player_df_r9, cnt_r9, max_player_r9 =  optimise_setup_fn(
        player_df_r5, player_df_r6, player_df_r7, player_df_r8, player_df_r9, efp_budget)

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

Change opt_strat to efp for player efp Optimisation Strategy


## Optimisation Runner

In [21]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    sel_player_df, sel_player_df_r5, sel_player_df_r6, sel_player_df_r7,sel_player_df_r8, sel_player_df_r9 = optimise_fn_efp(
        # Round 1
        # points_r1, price_r1, weight_r1, in_team_r1, available_r1, wk_weight_r1, bat_weight_r1, bowl_weight_r1,
        # play_cnt_r1, total_player_r1, wk_cnt_r1, total_wk_r1, bat_cnt_r1, total_bat_r1, bowl_cnt_r1, total_bowl_r1,
        # budget_r1, total_budget_r1, player_df_r1, cnt_r1, max_player_r1,
        # Round 2
        # points_r2, price_r2, weight_r2, in_team_r2, available_r2, wk_weight_r2, bat_weight_r2, bowl_weight_r2,
        # play_cnt_r2, total_player_r2, wk_cnt_r2, total_wk_r2, bat_cnt_r2, total_bat_r2, bowl_cnt_r2, total_bowl_r2,
        # budget_r2, total_budget_r2, team_play_cnt_r2, total_team_player_r2, player_df_r2, cnt_r2, max_player_r2,
        # Round 3
        # points_r3, price_r3, weight_r3, in_team_r3, available_r3, wk_weight_r3, bat_weight_r3, bowl_weight_r3,
        # play_cnt_r3, total_player_r3, wk_cnt_r3, total_wk_r3, bat_cnt_r3, total_bat_r3, bowl_cnt_r3, total_bowl_r3,
        # budget_r3, total_budget_r3, team_play_cnt_r3, total_team_player_r3, player_df_r3, cnt_r3, max_player_r3,
        # Round 4
        # points_r4, price_r4, weight_r4, in_team_r4, available_r4, wk_weight_r4, bat_weight_r4, bowl_weight_r4,
        # play_cnt_r4, total_player_r4, wk_cnt_r4, total_wk_r4, bat_cnt_r4, total_bat_r4, bowl_cnt_r4, total_bowl_r4,
        # budget_r4, total_budget_r4, team_play_cnt_r4, total_team_player_r4, player_df_r4, cnt_r4, max_player_r4,
        # Round 5
        points_r5, price_r5, weight_r5, in_team_r5, on_bench_r5, lock_f_r5, trade_in_r5, trade_out_r5, available_r5, wk_weight_r5, bat_weight_r5, bowl_weight_r5,
        play_cnt_r5, total_player_r5, wk_cnt_r5, total_wk_r5, bat_cnt_r5, total_bat_r5, bowl_cnt_r5, total_bowl_r5,
        budget_r5, total_budget_r5, team_play_cnt_r5, total_team_player_r5, player_df_r5, cnt_r5, max_player_r5,
        # Round 6
        points_r6, price_r6, weight_r6, in_team_r6, available_r6, wk_weight_r6, bat_weight_r6, bowl_weight_r6,
        play_cnt_r6, total_player_r6, wk_cnt_r6, total_wk_r6, bat_cnt_r6, total_bat_r6, bowl_cnt_r6, total_bowl_r6,
        budget_r6, total_budget_r6, team_play_cnt_r6, total_team_player_r6, player_df_r6, cnt_r6, max_player_r6,
        # Round 7
        points_r7, price_r7, weight_r7, in_team_r7, available_r7, wk_weight_r7, bat_weight_r7, bowl_weight_r7,
        play_cnt_r7, total_player_r7, wk_cnt_r7, total_wk_r7, bat_cnt_r7, total_bat_r7, bowl_cnt_r7, total_bowl_r7,
        budget_r7, total_budget_r7, team_play_cnt_r7, total_team_player_r7, player_df_r7, cnt_r7, max_player_r7,
        # Round 8
        points_r8, price_r8, weight_r8, in_team_r8, available_r8, wk_weight_r8, bat_weight_r8, bowl_weight_r8,
        play_cnt_r8, total_player_r8, wk_cnt_r8, total_wk_r8, bat_cnt_r8, total_bat_r8, bowl_cnt_r8, total_bowl_r8,
        budget_r8, total_budget_r8, team_play_cnt_r8, total_team_player_r8, player_df_r8, cnt_r8, max_player_r8,
        # Round 9
        points_r9, price_r9, weight_r9, in_team_r9, available_r9, wk_weight_r9, bat_weight_r9, bowl_weight_r9,
        play_cnt_r9, total_player_r9, wk_cnt_r9, total_wk_r9, bat_cnt_r9, total_bat_r9, bowl_cnt_r9, total_bowl_r9,
        budget_r9, total_budget_r9, team_play_cnt_r9, total_team_player_r9, player_df_r9, cnt_r9, max_player_r9
    )

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

Change opt_strat to efp for player efp Optimisation Strategy


## Optimisation Results

In [22]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    # Display Round-wise Selected Teams
    print("Round 5 Selected Team")
    display(sel_player_df_r5.sort_values(by='exp_rnd_points', ascending=False))

    # Display Traded In Players
    traded_in_r5 = sel_player_df_r5[sel_player_df_r5['In_Team'] == 0]
    print("Round 5 Traded In Players")
    display(traded_in_r5.sort_values(by='exp_rnd_points', ascending=False))

    # Display Traded Out Players
    traded_out_r5 = player_df_r5[player_df_r5['In_Team'] == 1][~player_df_r5['Name'].isin(sel_player_df_r5['Name'])].drop_duplicates(subset=['Name'])
    print("Round 5 Traded Out Players")
    display(traded_out_r5.sort_values(by='exp_rnd_points', ascending=False))

    # Save Optimal Team to CSV
    if exp_pts_model == 'all':
        sel_player_df.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_team_rnd_5_pg21.csv'), index=False)
        sel_player_df_r5.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_team_round_rnd_5_pg21.csv'), index=False)
        print("EFP Optimal Team Saved")
    elif exp_pts_model == 'bat_bowl':
        sel_player_df.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_team_dual_rnd_5_pg21.csv'), index=False)
        sel_player_df_r5.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_team_dual_round_rnd_5_pg21.csv'), index=False)
        print("EFP Bat + Bowl Optimal Team Saved")
else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

Change opt_strat to efp for player efp Optimisation Strategy


# 2. New Distribution Approach

# a. Data Extraction 

In [23]:
# New Optimisation Strategy (Player Fantasy Point Distribution)
if opt_strat == 'sim':
    # Data Extraction 
    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026_sim_rnd_4_pg21.csv'), low_memory=False)

    # Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team", "On_Bench", "Trade_In", "Trade_Out"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
    # team_fix_df = team_fix_df[team_fix_df.Round >= current_rnd].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df, team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected & std points csv file
    if exp_pts_model == 'bat_bowl':
        efp_raw_df = pd.read_csv(os.path.join(py_data_score_rnd_directory,'bbl15_efp_bat_bowl_model_score_short_pr4.csv'), low_memory=False)
        sfp_df = pd.read_csv(os.path.join(py_data_score_directory,'bbl15_sfp_model_score_short_upd3.csv'), low_memory=False)
        efp_df = efp_raw_df.merge(sfp_df, left_on = ["player", "Name", "Team", "opp", "venue", "Home_f","Round"], right_on = ["player", "Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")

    elif exp_pts_model == 'all':
        efp_raw_df = pd.read_csv(os.path.join(py_data_score_rnd_directory,'bbl15_efp_model_score_short_pr4.csv'), low_memory=False)
        sfp_df = pd.read_csv(os.path.join(py_data_score_directory,'bbl15_sfp_model_score_short_upd3.csv'), low_memory=False)
        efp_df = efp_raw_df.merge(sfp_df, left_on = ["player", "Name", "Team", "opp", "venue", "Home_f","Round"], right_on = ["player", "Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")

    # # Join on expected points for each game
    player_df_raw = pd.merge(game_df, efp_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")
    player_df_raw["exp_pts"] = np.where((player_df_raw["Role"] == "WK"), player_df_raw["exp_pts"] + 11.68, player_df_raw["exp_pts"] + 4.42)
    player_df_raw = player_df_raw.rename(columns={"exp_pts": "mean"})
    player_df_raw = player_df_raw.rename(columns={"sd_pts": "std_dev"})
    
    # Lock teams: Set Available=0 for players from locked teams in current round only (in team players should be unaffected)
    # Create flag for locked teams in current round
    player_df_raw['Lock_f'] = np.where((player_df_raw['Team'].isin(locked_teams)) & (player_df_raw['Round'] == current_rnd), 1, 0)
    print(f"Locked teams for Round {current_rnd}: {locked_teams}")

    # Set Available=0 for players from locked teams in current round only (in team players should be unaffected)
    player_df_raw.loc[(player_df_raw['Team'].isin(locked_teams)) & (player_df_raw['Round'] == current_rnd) & (player_df_raw['In_Team'] != 1), 'Available'] = 0

    # Set trade in players to available
    player_df_raw.loc[(player_df_raw['Trade_In'] == 1) & (player_df_raw['Round'] == current_rnd), 'Available'] = 1

    # Set trade out players to unlocked
    player_df_raw.loc[(player_df_raw['Trade_Out'] == 1) & (player_df_raw['Round'] == current_rnd), 'Lock_f'] = 0

    # Drop unnecessary columns
    # player_df_raw = player_df_raw[["Name", "Team", "Role", "Price", "Wk_f", "Bat_f", "Bowl_f", "Available", "In_Team","On_Bench","Trade_In", "Trade_Out", "Round", "opp", "venue", "Home_f","Lock_f", "exp_points"]]
    player_df_raw["weight"] = 1

    # Create game count per player
    player_df_raw["game_num"] = player_df_raw.groupby("Name").cumcount() + 1

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")


Locked teams for Round 5: ['Hobart Hurricanes', 'Sydney Thunder', 'Brisbane Heat', 'Melbourne Stars']


# b. Optimisation Process (During Tournament Selection) - Starting 12 Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 13 (Incl Bench Player)
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total starting budget of team is less than $15 player budget + surplass (bench costs $79,000)

In [24]:
# Sim FP Whole Tournament Optimisation
if opt_strat == 'sim':
    # Load Model Objects
    price_model_obj_1 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_1_game'))
    price_model_obj_2 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_2_game'))
    price_model_obj_3 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_3_game'))

    # Run Simulation Optimisation Function (with parallel execution enabled)
    all_sim_sel_players = optimise_fn_sim_fp(
        conf_int, sim_num, current_rnd, sim_budget, player_df_raw, price_df, 
        price_model_obj_1, price_model_obj_2, price_model_obj_3,
        use_parallel=True  # Set to False for sequential execution
    )

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")
 

For 20% simulated points confidence interval the lower z score is -0.253 and upper z score is 0.253
Completed 10/5000 simulations
Completed 20/5000 simulations
Completed 30/5000 simulations
Completed 40/5000 simulations
Completed 50/5000 simulations
Completed 60/5000 simulations
Completed 70/5000 simulations
Completed 80/5000 simulations
Completed 90/5000 simulations
Completed 100/5000 simulations
Completed 110/5000 simulations
Completed 120/5000 simulations
Completed 130/5000 simulations
Completed 140/5000 simulations
Completed 150/5000 simulations
Completed 160/5000 simulations
Completed 170/5000 simulations
Completed 180/5000 simulations
Completed 190/5000 simulations
Completed 200/5000 simulations
Completed 210/5000 simulations
Completed 220/5000 simulations
Completed 230/5000 simulations
Completed 240/5000 simulations
Completed 250/5000 simulations
Completed 260/5000 simulations
Completed 270/5000 simulations
Completed 280/5000 simulations
Completed 290/5000 simulations
Completed 

In [25]:
# Find Most Common Player Combinations Across Simulations
if opt_strat == 'sim':
    from collections import Counter
    
    # Group by simulation and create player combination sets
    sim_combinations = []
    
    for sim_id in all_sim_sel_players['Simulation'].unique():
        sim_players = all_sim_sel_players[all_sim_sel_players['Simulation'] == sim_id]['Name'].unique()
        # Create a frozenset to make it hashable for counting
        player_set = frozenset(sim_players)
        sim_combinations.append(player_set)
    
    # Count frequency of each combination
    combination_counts = Counter(sim_combinations)
    
    # Sort by frequency (descending)
    sorted_combinations = sorted(combination_counts.items(), key=lambda x: x[1], reverse=True)
    
    # Display results
    print(f"Total Unique Player Combinations: {len(combination_counts)}")
    print(f"Total Simulations: {len(sim_combinations)}\n")
    
    # Show top 10 combinations
    print("=" * 80)
    print("TOP 10 MOST COMMON PLAYER COMBINATIONS")
    print("=" * 80)
    
    for rank, (player_combo, count) in enumerate(sorted_combinations[:10], 1):
        percentage = (count / len(sim_combinations)) * 100
        print(f"\nRank {rank}: {count}/{len(sim_combinations)} simulations ({percentage:.1f}%)")
        print(f"Players: {sorted(list(player_combo))}")
        print("-" * 80)

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")

Total Unique Player Combinations: 18
Total Simulations: 5000

TOP 10 MOST COMMON PLAYER COMBINATIONS

Rank 1: 3078/5000 simulations (61.6%)
Players: ['Aaron Hardie', 'Brody Couch', 'Glenn Maxwell', 'Jack Wildermuth', 'Marcus Stoinis', 'Matthew Short', 'Matthew Wade', 'Mitch Owen', 'Mitchell Swepson', 'Nathan McAndrew', 'Peter Siddle', 'Riley Meredith', 'Sam Harper', 'Tom Curran', 'Will Sutherland']
--------------------------------------------------------------------------------

Rank 2: 1462/5000 simulations (29.2%)
Players: ['Aaron Hardie', 'Glenn Maxwell', 'Jack Wildermuth', 'Mahli Beardman', 'Marcus Stoinis', 'Matthew Short', 'Matthew Wade', 'Mitch Owen', 'Mitchell Swepson', 'Nathan McAndrew', 'Peter Siddle', 'Riley Meredith', 'Sam Harper', 'Tom Curran', 'Will Sutherland']
--------------------------------------------------------------------------------

Rank 3: 315/5000 simulations (6.3%)
Players: ['Aaron Hardie', 'Glenn Maxwell', 'Gurinder Sandhu', 'Jack Wildermuth', 'Marcus Stoini

In [26]:
# Create DataFrame Summary of Most Common Combinations
if opt_strat == 'sim':
    # Get current team players
    current_team_players = set(price_df[price_df['In_Team'] == 1]['Name'].tolist())
    
    # Create a dataframe with combination rankings
    combination_summary = []
    
    for rank, (player_combo, count) in enumerate(sorted_combinations, 1):
        percentage = (count / len(sim_combinations)) * 100
        
        # Find traded out players (in current team but not in this combination)
        traded_out_players = current_team_players - set(player_combo)
        
        # Find traded in players (in this combination but not in current team)
        traded_in_players = set(player_combo) - current_team_players
        
        combination_summary.append({
            'Rank': rank,
            'Frequency': count,
            'Percentage': f"{percentage:.1f}%",
            'Players': ', '.join(sorted(list(player_combo))),
            'Num_Players': len(player_combo),
            'Traded_In': ', '.join(sorted(list(traded_in_players))) if traded_in_players else 'None',
            'Traded_Out': ', '.join(sorted(list(traded_out_players))) if traded_out_players else 'None'
        })
    
    combination_summary_df = pd.DataFrame(combination_summary)
    
    # Display the top combinations as a nice table
    print("\nCOMBINATION SUMMARY TABLE:")
    print(combination_summary_df.head(10).to_string(index=False))
    
    # Get the most common combination
    if sorted_combinations:
        most_common_players = sorted(list(sorted_combinations[0][0]))
        most_common_count = sorted_combinations[0][1]
        most_common_percentage = (most_common_count / len(sim_combinations)) * 100
        
        print(f"\n{'='*80}")
        print(f"MOST COMMON COMBINATION SELECTED:")
        print(f"{'='*80}")
        print(f"Appears in: {most_common_count}/{len(sim_combinations)} simulations ({most_common_percentage:.1f}%)")
        print(f"\nPlayers ({len(most_common_players)}):")
        for i, player in enumerate(most_common_players, 1):
            print(f"  {i:2d}. {player}")
        print(f"{'='*80}")

    # Format most common players into dataframe
    most_common_players = pd.DataFrame(most_common_players, columns=['Name'])

    # Merge with player details
    most_common_players = pd.merge(most_common_players, price_df, on='Name', how='left')
    
    # Save Combination Summary to CSV
    if exp_pts_model == 'all':
        most_common_players.to_csv(os.path.join(add_data_directory,'optim/SFP_optimal_team_rnd_5_pg21.csv'), index=False)
        combination_summary_df.head(5).to_csv(os.path.join(add_data_directory,'optim/SFP_optimal_combos_rnd_5_pg21.csv'), index=False)

        print("Sim FP Combination Summary Saved")
    elif exp_pts_model == 'bat_bowl':
        most_common_players.to_csv(os.path.join(add_data_directory,'optim/SFP_optimal_team_dual_rnd_5_pg21.csv'), index=False)
        combination_summary_df.head(5).to_csv(os.path.join(add_data_directory,'optim/SFP_optimal_combos_dual_rnd_5_pg21.csv'), index=False)

        print("Sim FP Combination Summary Saved")

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")


COMBINATION SUMMARY TABLE:
 Rank  Frequency Percentage                                                                                                                                                                                                                          Players  Num_Players                                                   Traded_In                                            Traded_Out
    1       3078      61.6%     Aaron Hardie, Brody Couch, Glenn Maxwell, Jack Wildermuth, Marcus Stoinis, Matthew Short, Matthew Wade, Mitch Owen, Mitchell Swepson, Nathan McAndrew, Peter Siddle, Riley Meredith, Sam Harper, Tom Curran, Will Sutherland           15                  Mitchell Swepson, Peter Siddle, Sam Harper              Finn Allen, Mahli Beardman, Nathan Ellis
    2       1462      29.2%  Aaron Hardie, Glenn Maxwell, Jack Wildermuth, Mahli Beardman, Marcus Stoinis, Matthew Short, Matthew Wade, Mitch Owen, Mitchell Swepson, Nathan McAndrew, Peter Siddle, Riley Meredith,